# Workflow Testing Notebook

This notebook helps you test and improve your multi-agent workflow:
- Send a message through the full orchestrator
- Receive the final reply
- Inspect all workflow steps from the tracker
- Run each agent separately to compare quality and latency

In [1]:
import asyncio
import os
import sys
from datetime import datetime
from pathlib import Path
from time import perf_counter

import pandas as pd
from IPython.display import Markdown, display

try:
    from dotenv import load_dotenv
except Exception:
    load_dotenv = None

def _resolve_project_root() -> Path:
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    for p in candidates:
        if (p / "pyproject.toml").exists() and (p / "src").exists():
            return p
    return Path.cwd()

PROJECT_ROOT = _resolve_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Keep all relative paths (for secrets/docs) stable regardless of where notebook starts.
os.chdir(PROJECT_ROOT)

if load_dotenv is not None:
    load_dotenv(PROJECT_ROOT / ".env")

print(f"Project root: {PROJECT_ROOT}")
print(f"Current working directory: {Path.cwd()}")
print("Tip: make sure required API keys are set in environment/.env before running tests.")

Project root: /Users/nelsoncamoes/dev/personal/worldcup_2026
Current working directory: /Users/nelsoncamoes/dev/personal/worldcup_2026
Tip: make sure required API keys are set in environment/.env before running tests.


In [2]:
import importlib
import src.agents.orchestrator as _orch_mod
import src.agents.workflow_logger as _log_mod
import src.agents.planner_agent as _planner_mod
import src.agents.bigquery_agent as _bq_mod

importlib.reload(_log_mod)
importlib.reload(_planner_mod)
importlib.reload(_bq_mod)
importlib.reload(_orch_mod)

from src.agents.orchestrator import run_orchestrator
from src.agents.workflow_logger import get_tracker, reset_tracker

from src.agents.news_agent import run_structured as run_news
from src.agents.sentiment_agent import run_structured as run_sentiment
from src.agents.prediction_agent import run_structured as run_prediction
from src.agents.bigquery_agent import run_structured as run_bigquery

from langchain_openai import ChatOpenAI

print("Orchestrator + planner + bigquery reloaded.")

/Users/nelsoncamoes/dev/personal/worldcup_2026/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Orchestrator + planner + bigquery reloaded.


In [3]:
def _run_async(coro):
    """Runs async code safely from notebook context.

    In Jupyter, an event loop is often already running. In that case, run the
    coroutine in a separate thread with its own loop.
    """
    import concurrent.futures

    try:
        return asyncio.run(coro)
    except RuntimeError as exc:
        if "asyncio.run() cannot be called from a running event loop" not in str(exc):
            raise

        with concurrent.futures.ThreadPoolExecutor(max_workers=1) as executor:
            future = executor.submit(lambda: asyncio.run(coro))
            return future.result()


def run_workflow_test(
    user_message: str,
    user_id: str = "notebook_user",
    conversation_history: list[dict[str, str]] | None = None,
) -> dict:
    """Run the full orchestrator and return reply + workflow trace."""
    reset_tracker()
    t0 = perf_counter()
    reply = _run_async(
        run_orchestrator(
            user_message=user_message,
            user_id=user_id,
            conversation_history=conversation_history or [],
        )
    )
    total_ms = (perf_counter() - t0) * 1000

    steps = list(get_tracker().steps)
    return {
        "message": user_message,
        "reply": reply,
        "total_ms": round(total_ms, 2),
        "steps": steps,
    }


# ── Display helpers ───────────────────────────────────────────────────────

_STEP_ICONS = {
    "classify":        "🔎",
    "router":          "🗺️",
    "agent_execution": "⚙️",
    "aggregate":       "🔗",
    "confidence":      "📊",
    "compose":         "✍️",
}


def _hr():
    display(Markdown("---"))


def _section(title: str):
    display(Markdown(f"#### {title}"))


def _kv(label: str, value, indent: int = 0):
    pad = "&nbsp;" * (indent * 4)
    display(Markdown(f"{pad}**{label}:** {str(value)}"))


def _show_code_block(code: str, language: str = "sql"):
    display(Markdown(f"```{language}\n{code}\n```"))


def _show_nested(label: str, value, indent: int = 0):
    pad = "&nbsp;" * (indent * 4)

    if isinstance(value, dict):
        if not value:
            _kv(label, "{}", indent)
            return
        display(Markdown(f"{pad}**{label}:**"))
        for sub_key, sub_value in value.items():
            _show_nested(str(sub_key), sub_value, indent + 1)
        return

    if isinstance(value, list):
        if not value:
            _kv(label, "[]", indent)
            return
        if all(not isinstance(item, (dict, list)) for item in value):
            _kv(label, ", ".join(str(item) for item in value), indent)
            return
        display(Markdown(f"{pad}**{label}:**"))
        for idx, item in enumerate(value, 1):
            if isinstance(item, dict):
                display(Markdown(f"{'&nbsp;' * ((indent + 1) * 4)}**item {idx}:**"))
                for sub_key, sub_value in item.items():
                    _show_nested(str(sub_key), sub_value, indent + 2)
            else:
                _kv(f"item {idx}", item, indent + 1)
        return

    if label == "sql":
        display(Markdown(f"{pad}**sql:**"))
        _show_code_block(str(value), language="sql")
        return

    _kv(label, value, indent)


def _show_agent_outputs(agent_outputs: dict):
    for agent_name, payload in agent_outputs.items():
        if not isinstance(payload, dict):
            continue
        display(Markdown(f"**→ `{agent_name}`**"))
        _kv("data_source", payload.get("data_source", "unknown"), indent=1)
        _kv("confidence_score", payload.get("confidence_score"), indent=1)
        _kv("confidence_reason", payload.get("confidence_reason", ""), indent=1)

        metadata = payload.get("metadata") or {}
        if metadata:
            _show_nested("metadata", metadata, indent=1)

        answer = str(payload.get("answer", "")).strip()
        display(Markdown(f"&nbsp;&nbsp;&nbsp;&nbsp;**answer:**"))
        display(Markdown(answer or "_empty_"))


def _show_step(step: dict, delta_ms: float | None):
    node = step.get("node", "unknown")
    status = step.get("status", "")
    icon = _STEP_ICONS.get(node, "•")
    timing = f"  `+{delta_ms:.0f} ms`" if delta_ms is not None else ""
    display(Markdown(f"### {icon} `{node}` — {status}{timing}"))

    inp = step.get("input") or {}
    out = step.get("output") or {}
    meta = step.get("metadata") or {}

    if inp:
        _section("Inputs")
        for k, v in inp.items():
            if k == "user_message":
                display(Markdown("**user_message:**"))
                display(Markdown(f"> {v}"))
            else:
                _show_nested(k, v)

    if out:
        _section("Outputs")
        for k, v in out.items():
            if k == "agent_outputs":
                display(Markdown("**Per-agent results:**"))
                _show_agent_outputs(v)
            elif k in ("final_reply", "merged_answer", "answer"):
                display(Markdown(f"**{k}:**"))
                display(Markdown(str(v).strip()))
            else:
                _show_nested(k, v)

    if meta:
        _section("Metadata")
        for k, v in meta.items():
            _show_nested(k, v)

    _hr()


def show_workflow_result(result: dict) -> None:
    display(Markdown("# ✅ Final Reply"))
    display(Markdown(result["reply"]))
    display(Markdown(f"**Total latency:** `{result['total_ms']} ms`"))
    _hr()

    display(Markdown("# 🔬 Workflow Steps — Full Detail"))
    steps = result["steps"]
    prev_ts = None

    for step in steps:
        ts_raw = step.get("timestamp", "")
        ts = None
        if ts_raw:
            try:
                ts = datetime.fromisoformat(ts_raw)
            except Exception:
                pass

        delta_ms = None
        if ts is not None and prev_ts is not None:
            delta_ms = (ts - prev_ts).total_seconds() * 1000

        _show_step(step, delta_ms)

        if ts is not None:
            prev_ts = ts

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 3. DIAGNOSIS DASHBOARD — comprehensive per-run analysis
# ══════════════════════════════════════════════════════════════════════════════

def _extract_step_timing(steps: list[dict]) -> dict[str, float]:
    """Compute per-step latencies from step timestamps."""
    timings = {}
    prev_ts = None
    prev_name = None
    for step in steps:
        ts_raw = step.get("timestamp", "")
        name = step.get("node", "")
        if ts_raw and name:
            try:
                ts = datetime.fromisoformat(ts_raw)
            except Exception:
                continue
            if prev_ts and prev_name:
                delta = (ts - prev_ts).total_seconds() * 1000
                timings[name] = round(delta, 1)
                # Include preceding step's time too (time in previous node + time between)
                if prev_name not in timings:
                    timings[prev_name] = round(delta, 1)
            prev_ts = ts
            prev_name = name
    return timings


def _extract_agent_confidence(steps: list[dict]) -> dict[str, dict]:
    """Extract per-agent confidence from agent_execution step."""
    for step in steps:
        if step.get("node") != "agent_execution":
            continue
        out = step.get("output", {}) or {}
        agent_outputs = out.get("agent_outputs", {}) or {}
        result = {}
        for agent, payload in agent_outputs.items():
            if isinstance(payload, dict):
                result[agent] = {
                    "confidence": payload.get("confidence_score"),
                    "source": payload.get("data_source", "unknown"),
                    "reason": payload.get("confidence_reason", ""),
                }
        return result
    return {}


def _extract_fallback_flags(steps: list[dict]) -> list[str]:
    """Detect if any fallback paths were triggered."""
    flags = []
    for step in steps:
        out_str = str(step.get("output", ""))
        node = step.get("node", "")
        if "fallback" in out_str.lower():
            flags.append(f"'{node}' triggered fallback path")
        inp = step.get("input", {}) or {}
        if isinstance(inp, dict) and "error" in str(inp).lower():
            flags.append(f"'{node}' received error input")
    return flags


def _build_timing_waterfall(timings: dict[str, float], total_ms: float) -> str:
    """Build an ASCII waterfall chart."""
    if not timings:
        return "_(no timing data)_"
    lines = ["| Step | Time (ms) | % of Total |", "|------|-----------|------------|"]
    for name, ms in sorted(timings.items(), key=lambda x: x[1], reverse=True):
        pct = (ms / total_ms * 100) if total_ms else 0
        bar = "█" * int(pct / 2)
        lines.append(f"| `{name}` | {ms:.0f} | {pct:.0f}% {bar} |")
    return "\n".join(lines)


def _check_catalog_health() -> dict:
    """Quick health check on the BQ catalog."""
    from src.data.datamodel.catalog import list_tables, agent_visible_table_names, get_table
    from src.agents.planner_agent import AVAILABLE_AGENTS

    all_tables = list_tables(agent_visible=True)
    names = agent_visible_table_names()

    issues = []
    mart_count = sum(1 for t in all_tables if t.layer == "mart")
    fact_count = sum(1 for t in all_tables if t.layer == "fact")
    dim_count = sum(1 for t in all_tables if t.layer == "dim")

    # Check planner agents vs orchestrator runners
    orchestrator_agents = ["news", "sentiment", "prediction", "bigquery", "chat", "match_facts"]
    planner_missing = [a for a in orchestrator_agents if a not in AVAILABLE_AGENTS]
    if planner_missing:
        issues.append(f"⚠️ Agents in orchestrator but NOT in planner: {planner_missing}")

    # Check preferred marts exist
    preferred_marts = [t for t in all_tables if t.preferred]
    if not preferred_marts:
        issues.append("⚠️ No preferred marts found in catalog")

    return {
        "table_count": len(names),
        "mart_count": mart_count,
        "fact_count": fact_count,
        "dim_count": dim_count,
        "planner_agents": AVAILABLE_AGENTS,
        "planner_missing_agents": planner_missing,
        "preferred_marts": [t.name for t in preferred_marts],
        "issues": issues,
    }


def show_diagnosis_dashboard(result: dict) -> None:
    """Full diagnosis dashboard: timing, confidence, fallbacks, catalog health."""
    steps = result.get("steps", [])
    total_ms = result.get("total_ms", 0)

    # ── Section 1: Overview ──
    display(Markdown("# 🩺 Diagnosis Dashboard"))

    display(Markdown("## 📈 Execution Summary"))
    display(Markdown(f"| Metric | Value |"))
    display(Markdown(f"|--------|-------|"))
    display(Markdown(f"| Total latency | **{total_ms:.0f} ms** |"))
    display(Markdown(f"| Nodes executed | {len(steps)} |"))

    for step in steps:
        status = step.get("status", "?")
        emoji = "✅" if status == "executed" else "❌"
        display(Markdown(f"| {emoji} `{step.get('node', '?')}` | {status} |"))

    # ── Section 2: Timing Waterfall ──
    timings = _extract_step_timing(steps)
    display(Markdown("## ⏱️ Timing Waterfall"))
    display(Markdown(_build_timing_waterfall(timings, total_ms)))

    # Highlight slowest step
    if timings:
        slowest = max(timings.items(), key=lambda x: x[1])
        if slowest[1] > total_ms * 0.5:
            display(Markdown(f"🐌 **Bottleneck:** `{slowest[0]}` takes {slowest[1]:.0f} ms ({slowest[1]/total_ms*100:.0f}% of total)"))

    # ── Section 3: Agent Confidence ──
    confidences = _extract_agent_confidence(steps)
    if confidences:
        display(Markdown("## 🎯 Agent Confidence Scores"))
        for agent, info in confidences.items():
            score = info.get("confidence") or 0
            pct = int(score * 100) if score else 0
            bar = "█" * (pct // 5) + "░" * (20 - pct // 5)
            label = "🟢 HIGH" if pct >= 80 else ("🟡 MED" if pct >= 55 else "🔴 LOW")
            display(Markdown(f"**`{agent}`** — {label} `{pct}%`"))
            display(Markdown(f"&nbsp;&nbsp;&nbsp;`{bar}`"))
            display(Markdown(f"&nbsp;&nbsp;&nbsp;Source: `{info.get('source', '?')}` | {info.get('reason', '')}"))

    # ── Section 4: Fallback Detection ──
    fallbacks = _extract_fallback_flags(steps)
    if fallbacks:
        display(Markdown("## ⚠️ Fallback Paths Triggered"))
        for f in fallbacks:
            display(Markdown(f"- {f}"))
    else:
        display(Markdown("## ✅ No Fallbacks"))
        display(Markdown("All agents executed on their primary path."))

    # ── Section 5: Catalog Health ──
    display(Markdown("## 🏥 Catalog Health Check"))
    try:
        health = _check_catalog_health()
        display(Markdown(f"| Check | Status |"))
        display(Markdown(f"|-------|--------|"))
        display(Markdown(f"| Agent-visible tables | **{health['table_count']}** ({health['mart_count']} marts, {health['fact_count']} facts, {health['dim_count']} dims) |"))

        if health["planner_missing_agents"]:
            display(Markdown(f"| ⚠️ Planner missing agents | **{health['planner_missing_agents']}** — not routable! |"))
        else:
            display(Markdown(f"| Planner ↔ Orchestrator sync | ✅ All agents routable |"))

        if health["issues"]:
            for issue in health["issues"]:
                display(Markdown(f"| ⚠️ Issue | {issue} |"))
        else:
            display(Markdown(f"| Catalog integrity | ✅ No issues |"))
    except Exception as exc:
        display(Markdown(f"⚠️ Catalog health check failed: {exc}"))

    # ── Section 6: Planner Decision Analysis ──
    display(Markdown("## 🧠 Planner Decision"))
    for step in steps:
        if step.get("node") == "router":
            out = step.get("output", {}) or {}
            display(Markdown(f"- **Selected agents:** `{out.get('selected_agents', [])}`"))
            display(Markdown(f"- **Response mode:** `{out.get('response_mode', '?')}`"))
            display(Markdown(f"- **Reasoning:** {out.get('planner_reason', '_none_')}"))

    # ── Section 7: Intent Classification ──
    for step in steps:
        if step.get("node") == "classify":
            out = step.get("output", {}) or {}
            display(Markdown("## 🏷️ Intent Classification"))
            display(Markdown(f"**Intent:** `{out.get('intent', '?')}`"))

    display(Markdown("---"))


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 2. BIGQUERY SQL INSPECTOR — extracts & displays every SQL query executed
# ══════════════════════════════════════════════════════════════════════════════

def extract_bigquery_sql(result: dict) -> list[dict]:
    """Pull all BigQuery SQL statements + metadata from a workflow result."""
    queries = []
    steps = result.get("steps", [])

    # Method 1: from agent_execution step outputs (agent_outputs contains metadata.sql_executed)
    for step in steps:
        out = step.get("output", {}) or {}

        # Direct agent_outputs from the execute_agents node
        agent_outputs = out.get("agent_outputs", {}) or {}
        for agent_name, payload in agent_outputs.items():
            if not isinstance(payload, dict):
                continue
            metadata = payload.get("metadata", {}) or {}
            sql_list = metadata.get("sql_executed", [])
            if isinstance(sql_list, (list, tuple)):
                for sql in sql_list:
                    if isinstance(sql, str) and sql.strip():
                        queries.append({
                            "source": f"agent_outputs.{agent_name}.metadata.sql_executed",
                            "agent": agent_name,
                            "sql": sql.strip(),
                            "context": metadata,
                        })

        # Also check per-agent metadata for tool_calls (raw tool trace)
        for agent_name, payload in agent_outputs.items():
            if not isinstance(payload, dict):
                continue
            metadata = payload.get("metadata", {}) or {}
            tool_calls = metadata.get("tool_calls", [])
            for tc in tool_calls:
                if isinstance(tc, dict) and tc.get("tool") == "run_sql":
                    args = tc.get("args", {}) or {}
                    sql = args.get("sql", "")
                    if isinstance(sql, str) and sql.strip():
                        queries.append({
                            "source": f"agent_outputs.{agent_name}.metadata.tool_calls",
                            "agent": agent_name,
                            "sql": sql.strip(),
                            "args": {k: v for k, v in args.items() if k != "sql"},
                        })

    # Method 2: from aggregate step output
    for step in steps:
        if step.get("node") != "aggregate":
            continue
        out = step.get("output", {}) or {}
        merged = out.get("merged_answer", "")
        agent_outputs = out.get("agent_outputs", {}) or {}

    return queries


def show_bigquery_queries(result: dict) -> None:
    """Display all BigQuery SQL queries executed in a run."""
    queries = extract_bigquery_sql(result)

    display(Markdown("## 📊 BigQuery SQL Executed"))
    if not queries:
        display(Markdown("ℹ️ _No BigQuery SQL queries were executed in this workflow._"))
        display(Markdown("---"))
        return

    # Deduplicate (same SQL, same agent)
    seen = set()
    unique = []
    for q in queries:
        key = (q["sql"], q["agent"])
        if key not in seen:
            seen.add(key)
            unique.append(q)

    display(Markdown(f"**{len(unique)} unique query(ies) from {len(queries)} total call(s)**"))

    for i, q in enumerate(unique, 1):
        agent = q["agent"]
        source = q["source"]
        agent_icon = {"bigquery": "📊", "prediction": "🔮", "match_facts": "📋"}.get(agent, "⚙️")

        display(Markdown(f"### {agent_icon} Query {i} — `{agent}`"))
        display(Markdown(f"_source: `{source}`_"))

        # SQL block
        display(Markdown(f"```sql\n{q['sql']}\n```"))

        # Additional context
        context = q.get("context", {})
        if context:
            row_count = context.get("row_count")
            error = context.get("error")
            if row_count is not None:
                display(Markdown(f"📈 **Rows returned:** {row_count}"))
            if error:
                display(Markdown(f"⚠️ **Error:** {error}"))

        if i < len(unique):
            display(Markdown("---"))

    display(Markdown("---"))


# Also: standalone BQ agent trace viewer (run bigquery_agent directly and see its raw tool calls)
def run_bigquery_raw(query: str) -> dict:
    """Run bigquery_agent directly and return the full trace (tool calls, SQL, row counts)."""
    from src.agents.bigquery_agent import _run_agent, _confidence

    t0 = perf_counter()
    try:
        answer, trace = _run_agent(query)
        score, reason = _confidence(trace)
        error = None
    except Exception as exc:
        answer = f"Error: {exc}"
        trace = []
        score, reason = 0.2, str(exc)
        error = str(exc)

    total_ms = round((perf_counter() - t0) * 1000, 2)

    sql_calls = []
    for t in trace:
        if t.get("tool") == "run_sql":
            sql_calls.append({
                "sql": (t.get("args") or {}).get("sql", ""),
                "row_count": t.get("row_count"),
                "error": t.get("error"),
            })

    return {
        "query": query,
        "answer": answer,
        "confidence_score": score,
        "confidence_reason": reason,
        "latency_ms": total_ms,
        "error": error,
        "tool_calls": [{"tool": t["tool"], "args": t.get("args")} for t in trace],
        "sql_executed": sql_calls,
    }


def show_bigquery_trace(bq_result: dict) -> None:
    """Display a full bigquery_agent run trace."""
    display(Markdown("## 🔬 BigQuery Agent — Full Trace"))
    display(Markdown(f"**Latency:** `{bq_result['latency_ms']} ms` | **Confidence:** {bq_result['confidence_score']:.0%}"))
    display(Markdown(f"**Reason:** {bq_result['confidence_reason']}"))

    tool_calls = bq_result.get("tool_calls", [])
    if not tool_calls:
        display(Markdown("ℹ️ _No tool calls made — agent answered directly._"))
    else:
        for i, tc in enumerate(tool_calls, 1):
            tool = tc["tool"]
            args = tc.get("args") or {}
            display(Markdown(f"**Turn {i}: `{tool}`**"))
            if tool == "run_sql":
                display(Markdown(f"```sql\n{args.get('sql', '')}\n```"))
            else:
                for k, v in args.items():
                    display(Markdown(f"- `{k}`: {str(v)[:200]}"))

    display(Markdown("---"))


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 1. FLOW DIAGRAM — rendered agent DAG that updates per prompt
# ══════════════════════════════════════════════════════════════════════════════

from IPython.display import display, HTML
import base64

try:
    import graphviz
    _HAS_GRAPHVIZ = True
except Exception:
    _HAS_GRAPHVIZ = False


def _build_mermaid_diagram(result: dict) -> str:
    """Build a Mermaid flowchart showing the actual agent execution path."""
    steps = result.get("steps", [])
    message = result.get("message", "")

    # Determine which agents were selected
    selected = []
    for s in steps:
        if s.get("node") == "router":
            selected = (s.get("output", {}) or {}).get("selected_agents", [])

    # Determine which agents actually executed
    executed = []
    for s in steps:
        if s.get("node") == "agent_execution":
            executed = (s.get("output", {}) or {}).get("executed_agents", [])

    agent_source_map = {}
    for s in steps:
        if s.get("node") == "agent_execution":
            outputs = (s.get("output", {}) or {}).get("agent_outputs", {}) or {}
            for agent, payload in outputs.items():
                ds = payload.get("data_source", "unknown") if isinstance(payload, dict) else "unknown"
                agent_source_map[agent] = ds

    # Build Mermaid
    lines = ["flowchart TD"]
    lines.append("    USER[👤 User Message]:::user")

    # Determine if classify was used
    intent = ""
    for s in steps:
        if s.get("node") == "classify":
            intent = (s.get("output", {}) or {}).get("intent", "")
    lines.append(f'    CLASSIFY[🔎 classify_intent<br/><i>intent: {intent}</i>]:::node')

    # Planner / Router
    planner_agents = ", ".join(selected) if selected else "chat"
    lines.append(f'    PLANNER[🗺️ route_request<br/><i>selected: {planner_agents}</i>]:::node')

    # Agent execution — show per-agent boxes
    agent_node_ids = []
    for agent in executed:
        node_id = f"AGENT_{agent.upper()}"
        ds = agent_source_map.get(agent, "unknown")
        icon_map = {"bigquery": "📊", "news": "📰", "sentiment": "💭", "prediction": "🔮", "chat": "💬", "match_facts": "📋"}
        icon = icon_map.get(agent, "⚙️")
        lines.append(f'    {node_id}[{icon} {agent}<br/><i>source: {ds}</i>]:::agent')
        agent_node_ids.append(node_id)

    # Aggregate
    has_multi = len(executed) > 1
    agg_label = "🔗 aggregate_outputs<br/><i>synthesize multi-agent</i>" if has_multi else "🔗 aggregate_outputs<br/><i>pass-through</i>"
    lines.append(f'    AGG[{agg_label}]:::node')

    # Confidence
    for s in steps:
        if s.get("node") == "confidence":
            out = s.get("output", {}) or {}
            label = out.get("confidence_label", "medium").upper()
            score = out.get("confidence_score", 0)
    conf_style = "conf_high" if label == "HIGH" else ("conf_low" if label == "LOW" else "conf_med")
    lines.append(f'    CONF[📊 score_confidence<br/><i>{label} ({score*100:.0f}%)</i>]:::{{conf_style}}')

    # Compose
    lines.append(f'    COMPOSE[✍️ compose_reply<br/><i>format final answer</i>]:::node')
    lines.append(f'    REPLY[✅ Final Reply<br/><i>{result.get("total_ms", 0)} ms</i>]:::reply')

    # Edges
    lines.append("    USER --> CLASSIFY")
    lines.append("    CLASSIFY --> PLANNER")
    for nid in agent_node_ids:
        lines.append(f"    PLANNER --> {nid}")
    for nid in agent_node_ids:
        lines.append(f"    {nid} --> AGG")
    lines.append("    AGG --> CONF")
    lines.append("    CONF --> COMPOSE")
    lines.append("    COMPOSE --> REPLY")

    # Styles
    lines.append("    classDef user fill:#e1f5fe,stroke:#0288d1,color:#000")
    lines.append("    classDef node fill:#f3e5f5,stroke:#7b1fa2,color:#000")
    lines.append("    classDef agent fill:#fff3e0,stroke:#f57c00,color:#000")
    lines.append("    classDef conf_high fill:#c8e6c9,stroke:#388e3c,color:#000")
    lines.append("    classDef conf_med fill:#fff9c4,stroke:#fbc02d,color:#000")
    lines.append("    classDef conf_low fill:#ffcdd2,stroke:#d32f2f,color:#000")
    lines.append("    classDef reply fill:#e0f2f1,stroke:#00796b,color:#000")

    return "\n".join(lines)


def render_flow_diagram(result: dict) -> None:
    """Render the agent flow as both Mermaid HTML and (if available) Graphviz."""
    display(Markdown("## 🗺️ Agent Flow Diagram"))
    mermaid = _build_mermaid_diagram(result)

    # Mermaid via HTML (works in most Jupyter environments)
    mermaid_html = f"""<div class="mermaid">{mermaid}</div>
<script>
if (typeof mermaid !== 'undefined') {{
    mermaid.initialize({{ startOnLoad: true, theme: 'default' }});
    mermaid.run();
}}
</script>"""
    display(HTML(mermaid_html))

    # Also show raw Mermaid for copy-paste into mermaid.live
    display(Markdown("<details><summary>📋 Raw Mermaid (click to expand)</summary>"))
    display(Markdown(f"\n```mermaid\n{mermaid}\n```\n"))
    display(Markdown("</details>"))

    if _HAS_GRAPHVIZ:
        try:
            dot = graphviz.Digraph("agent_flow", format="png")
            dot.attr(rankdir="TB", bgcolor="white")
            dot.node("user", "👤 User Message", shape="box", style="filled", fillcolor="#e1f5fe")

            steps = result.get("steps", [])
            node_names = []
            for s in steps:
                n = s.get("node", "")
                if n:
                    node_names.append(n)
                    dot.node(n, n, shape="box", style="filled", fillcolor="#f3e5f5")

            if node_names:
                dot.edge("user", node_names[0])
                for i in range(len(node_names) - 1):
                    dot.edge(node_names[i], node_names[i + 1])

            display(dot)
        except Exception:
            pass

    display(Markdown("---"))


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# FULL WORKFLOW TEST — now with Flow Diagram + SQL Inspector + Diagnosis
# ══════════════════════════════════════════════════════════════════════════════

def run_full_diagnosis(
    user_message: str,
    user_id: str = "notebook_user",
    conversation_history: list[dict[str, str]] | None = None,
) -> dict:
    """Run the orchestrator and produce a complete diagnosis."""
    result = run_workflow_test(user_message, user_id=user_id, conversation_history=conversation_history)

    # 1. Show final reply
    display(Markdown("# ✅ Final Reply"))
    display(Markdown(result["reply"]))
    display(Markdown(f"**Total latency:** `{result['total_ms']} ms`"))

    # 2. Rendered flow diagram
    render_flow_diagram(result)

    # 3. BigQuery SQL inspector
    show_bigquery_queries(result)

    # 4. Full diagnosis dashboard
    show_diagnosis_dashboard(result)

    # 5. Raw workflow steps (collapsed by default)
    display(Markdown("<details><summary>🔬 Raw Workflow Steps (click to expand)</summary>"))
    show_workflow_result(result)
    display(Markdown("</details>"))

    return result


# ── Run the test ──
test_message = "What was Portugal vs Morocco last result and stats?"
result = run_full_diagnosis(test_message, user_id="notebook_user")


# ✅ Final Reply

## Portugal vs Morocco Last Result

- **Date**: December 10, 2022
- **Competition**: World Cup, Quarter-finals
- **Venue**: Al Thumama Stadium, Doha
- **Final Score**: Morocco 1 - 0 Portugal
- **Referee**: F. Tello
- **Status**: Full Time

## Match Stats

### Morocco
- **Shots on Goal**: 3
- **Ball Possession**: 27%
- **Shots Off Goal**: 6
- **Total Shots**: 9
- **Corner Kicks**: 3
- **Fouls**: 15

### Portugal
- **Shots on Goal**: 3
- **Ball Possession**: 73%
- **Shots Off Goal**: 6
- **Total Shots**: 12
- **Corner Kicks**: 9
- **Fouls**: 9

Confidence: HIGH (80%)

**Total latency:** `19661.33 ms`

---

# 🔬 Workflow Steps — Full Detail

...

**intent:** data

#### Outputs

**final_reply:**

## Portugal vs Morocco Last Result

- **Date**: December 10, 2022
- **Competition**: World Cup, Quarter-finals
- **Venue**: Al Thumama Stadium, Doha
- **Final Score**: Morocco 1 - 0 Portugal
- **Referee**: F. Tello
- **Status**: Full Time

## Match Stats

### Morocco
- **Shots on Goal**: 3
- **Ball Possession**: 27%
- **Shots Off Goal**: 6
- **Total Shots**: 9
- **Corner Kicks**: 3
- **Fouls**: 15

### Portugal
- **Shots on Goal**: 3
- **Ball Possession**: 73%
- **Shots Off Goal**: 6
- **Total Shots**: 12
- **Corner Kicks**: 9
- **Fouls**: 9

Confidence: HIGH (80%)

---

In [ ]:
# Optional conversation history simulation
conversation = [
    {"role": "user", "content": "Show me Portugal next fixture"},
    {"role": "assistant", "content": "Portugal plays Team X on ..."},
]
follow_up = "And what are the win probabilities for that one?"
follow_up_result = run_workflow_test(follow_up, conversation_history=conversation)
show_workflow_result(follow_up_result)

In [ ]:
# match_facts_agent is no longer routed by the orchestrator.
# bigquery_agent is now the single source of truth for all structured data.
AGENT_RUNNERS = {
    "news": run_news,
    "sentiment": run_sentiment,
    "prediction": run_prediction,
    "bigquery": run_bigquery,
}

def run_chat_agent(query: str) -> dict:
    """Simple isolated chat baseline using the same LLM family."""
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    prompt = (
        "You are a helpful football assistant. Answer concisely and clearly.\n"
        f"User: {query}"
    )
    answer = llm.invoke(prompt).content.strip()
    return {
        "answer": answer,
        "confidence_score": 0.7,
        "confidence_reason": "Direct chat baseline response.",
        "metadata": {"path": "isolated_chat"},
    }

AGENT_RUNNERS["chat"] = run_chat_agent

def run_single_agent_test(agent_name: str, query: str) -> dict:
    if agent_name not in AGENT_RUNNERS:
        raise ValueError(f"Unknown agent: {agent_name}. Available: {list(AGENT_RUNNERS)}")

    t0 = perf_counter()
    error = None
    payload = {}
    try:
        payload = AGENT_RUNNERS[agent_name](query)
    except Exception as exc:
        error = str(exc)
    total_ms = round((perf_counter() - t0) * 1000, 2)

    return {
        "agent": agent_name,
        "query": query,
        "latency_ms": total_ms,
        "error": error,
        "payload": payload,
    }

def compare_agents(query: str, agents: list[str] | None = None) -> pd.DataFrame:
    selected = agents or list(AGENT_RUNNERS.keys())
    rows = []
    for name in selected:
        out = run_single_agent_test(name, query)
        payload = out.get("payload") or {}
        rows.append({
            "agent": name,
            "latency_ms": out["latency_ms"],
            "confidence_score": payload.get("confidence_score"),
            "confidence_reason": payload.get("confidence_reason"),
            "answer_preview": str(payload.get("answer", ""))[:140],
            "error": out.get("error"),
        })
    return pd.DataFrame(rows).sort_values(by="latency_ms", ascending=True)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# ISOLATED AGENT TESTING — with deep BQ trace
# ══════════════════════════════════════════════════════════════════════════════

query = "Portugal vs Morocco World Cup 2026"

# 1. Compare all agents
display(Markdown("## 🆚 Agent Comparison"))
agent_df = compare_agents(query, agents=["news", "sentiment", "prediction", "bigquery", "chat"])
display(agent_df)

# 2. Deep dive bigquery agent — raw trace
display(Markdown("## 🔬 BigQuery Agent — Raw Trace"))
bq_deep = run_bigquery_raw(query)
show_bigquery_trace(bq_deep)

# 3. Also run the standard single-agent test for comparison
single = run_single_agent_test("bigquery", query)
display(Markdown("## 📊 BigQuery Agent — Structured Output"))
display(Markdown(f"- **Confidence:** {single['payload'].get('confidence_score', '?')}"))
display(Markdown(f"- **Reason:** {single['payload'].get('confidence_reason', '?')}"))
display(Markdown(f"- **Latency:** {single['latency_ms']} ms"))
display(Markdown(f"- **Error:** {single.get('error') or 'none'}"))
display(Markdown("**Answer:**"))
display(Markdown((single["payload"] or {}).get("answer", "")[:2000]))


## How To Use This For Workflow Improvement

### Quick Start
1. **Run Cell 5** (`run_full_diagnosis`) — this is your main testing cell. Change `test_message` to any prompt.
2. Every run produces:
   - ✅ **Final reply** — what the user would see
   - 🗺️ **Flow diagram** — rendered Mermaid chart showing which agents ran
   - 📊 **BigQuery SQL** — every SQL statement executed, with row counts
   - 🩺 **Diagnosis Dashboard** — timing waterfall, confidence scores, fallback detection, catalog health, planner decisions

### What to Diagnose Per Run

| Diagnosis | What to Look For | Action |
|-----------|-----------------|--------|
| **Flow Diagram** | Wrong agent selected? Missing `match_facts`? | Fix planner rules or `AVAILABLE_AGENTS` |
| **SQL Inspector** | SQL errors? Missing `LIMIT`? Wrong table? | Fix catalog metadata or BQ agent system prompt |
| **Timing Waterfall** | One step >50% of total | Optimize that step (cache, parallelize, faster model) |
| **Confidence Scores** | LOW confidence when answer seems good? | Adjust confidence thresholds or reason logic |
| **Fallback Detection** | Fallback triggered unnecessarily? | Improve primary path (data freshness, query parsing) |
| **Catalog Health** | Planner missing agents? | Add to `planner_agent.py:AVAILABLE_AGENTS` |
| **Planner Decision** | Wrong agents selected for query type? | Refine planner prompt or fallback rules |

### Iterative Refinement Loop
1. Write a representative `test_message`
2. Run `run_full_diagnosis(test_message)`
3. Inspect all 4 diagnostic panels
4. Identify the weakest link (slowest step, lowest confidence, wrong routing)
5. Fix the code (agent prompt, planner rules, catalog metadata, data model)
6. Re-run — did the diagnosis improve?
7. Repeat until confidence ≥ 0.8 and latency < 5s for all target question types

### Recommended Test Suite
Run these prompts and check that each routes to the right agents:
- `"What matches are played today?"` → should route to `bigquery` (or `match_facts`)
- `"Who will win Portugal vs Morocco?"` → should route to `prediction` + `bigquery`
- `"What's the latest news about Ronaldo?"` → should route to `news`
- `"How are fans feeling about Brazil?"` → should route to `sentiment`
- `"Hello!"` → should route to `chat`
- `"Show me Argentina's last 5 results"` → should route to `bigquery`

### Advanced: BQ Agent Isolation
Use Cell 8 to run `run_bigquery_raw()` directly — this bypasses the orchestrator and shows every tool call the BQ agent makes (list_tables, describe_table, sample_table, run_sql). Useful for debugging:
- Is the agent picking the right mart?
- Is it calling `describe_table` before writing SQL?
- Are queries failing due to column name mismatches?
